# DeciderBench × Jev — evaluation of a calibrated System One decision model

This notebook runs the **DeciderBench eval set** (7,287 one-question decision tasks, 11 source datasets)
against TypeSafe's **Jev** System One API and reports the metrics that matter for an
**RLCD**-trained (Reinforcement Learning for Calibrated Decisions) model:

| Area | Metrics |
|---|---|
| Correctness | accuracy (+95% Wilson CI) **per dataset**, per primitive, per BBH task; balanced accuracy, macro-F1, majority-class baseline |
| Proper scoring rules (RLCD reward) | log loss / NLL (log scoring rule), Brier score (multi-class Σ form and the article's 1/K form) |
| Calibration | top-label ECE (10 equal-width bins), equal-mass ECE, MCE, over/under-confidence gap, reliability diagrams |
| Using confidence | AUROC of confidence for correctness, risk–coverage (AURC), coverage/accuracy at thresholds, the article's decision-theoretic auto-route threshold τ = (C_err − C_human)/(C_err − C_auto) with expected cost per decision |
| Primitive-specific | Score: MAE of expected score, ranked probability score, ±1 accuracy · Noul: AUROC, AUPRC, positive-class F1 |
| Operations | requests, errors, latency p50/p95/p99, input tokens, $ cost at $0.04 / 1M input tokens |

**How to run:** set the config cell (API key source, S3 paths), then *Run All*. The API stage is resumable:
raw responses are cached to `OUT_DIR/responses.jsonl`, and re-running skips requests already answered.
Everything (report, tables, figures, raw responses, this notebook) is uploaded to `S3_REPORT_PREFIX`.

**Network requirements:** HTTPS to `api.typesafe.ai` (or your `TYPESAFE_BASE_URL`) and to S3.

**Gold labels never come from Jev**; they are the source datasets' labels carried in each eval row.

In [ ]:
# ============================== CONFIG ==============================
import datetime as _dt
import os

# --- data ---
LOCAL_EVAL_PATH = "/Users/dhruvilpatel/Downloads/D/decision_model_bench/DeciderBench/DeciderBench_eval.jsonl"          # downloaded here if missing
API_KEY = "API_KEY"

# --- API ---
BASE_URL = os.environ.get("TYPESAFE_BASE_URL", "https://api.typesafe.ai").rstrip("/")
MODEL = "jev-latest"
WORKERS = 8                  # concurrent requests
TIMEOUT_S = 120
MAX_ATTEMPTS = 8             # retries on 429/529/5xx/network errors with exponential backoff
BATCH_SAME_STATE = True      # rows with an identical state go in one request (questions are independent)
LIMIT_REQUESTS = None        # e.g. 50 for a smoke test; None = full eval set
MOCK = False                 # True = synthetic responses (pipeline test without the API; NOT real results)

# --- pricing / routing economics (article example values) ---
PRICE_PER_M_INPUT = 0.04     # USD per 1M input tokens (output tokens are free)
C_AUTO, C_HUMAN, C_ERROR = 0.05, 1.00, 5.00
CONF_THRESHOLDS = [0.5, 0.7, 0.81, 0.9, 0.95]
ECE_BINS = 10

# --- outputs ---
RUN_ID = _dt.datetime.utcnow().strftime("%Y%m%dT%H%M%SZ") + ("_MOCK" if MOCK else "")
OUT_DIR = f"jev_eval_runs/{MODEL}"                  # responses cache lives here (shared across runs)
REPORT_DIR = f"{OUT_DIR}/report_{RUN_ID}"
S3_REPORT_PREFIX = f"s3://datasets-exp/DeciderBench/jev_eval/{RUN_ID}/"
NOTEBOOK_PATH = "DeciderBench_jev_eval.ipynb"       # uploaded alongside the report if it exists
UPLOAD_TO_S3 = True

/var/folders/w3/591d6_x15lq37kf4qn3t_7dc0000gn/T/ipykernel_65584/2422538811.py:26: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  RUN_ID = _dt.datetime.utcnow().strftime("%Y%m%dT%H%M%SZ") + ("_MOCK" if MOCK else "")


In [2]:
# Dependencies (no-op if already installed)
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "pandas>=1.3", "numpy>=1.21", "matplotlib>=3.4", "pyarrow>=8", "boto3>=1.26"], check=False)

CompletedProcess(args=['/Users/dhruvilpatel/Downloads/D/decision_model_bench/.venv/bin/python', '-m', 'pip', 'install', '-q', 'pandas>=1.3', 'numpy>=1.21', 'matplotlib>=3.4', 'pyarrow>=8', 'boto3>=1.26'], returncode=0)

In [3]:
import base64, collections, concurrent.futures as cf, getpass, hashlib, io, json, math, random, shutil
import socket, threading, time, urllib.error, urllib.parse, urllib.request
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

pd.set_option("display.width", 200, "display.max_columns", 40, "display.max_rows", 200)
Path(OUT_DIR).mkdir(parents=True, exist_ok=True)
Path(REPORT_DIR, "figures").mkdir(parents=True, exist_ok=True)

rows = [json.loads(l) for l in open(LOCAL_EVAL_PATH, encoding="utf-8")]
eval_df = pd.DataFrame(rows)
print(f"{len(rows)} eval rows from {LOCAL_EVAL_PATH}")
display(eval_df.groupby(["dataset", "primitive"]).size().unstack(fill_value=0))

Matplotlib is building the font cache; this may take a moment.


7287 eval rows from /Users/dhruvilpatel/Downloads/D/decision_model_bench/DeciderBench/DeciderBench_eval.jsonl


primitive,choice,noul,score
dataset,,,
BBH,450,250,0
Belebele,500,0,0
CIRCA,500,0,0
ContractNLI,500,0,0
FinancialPhraseBank,0,0,1000
JevBench-Hard,94,128,71
JudgeBench,300,0,0
RAGTruth,0,1494,0
TabFact,0,500,0


In [4]:
# Build requests. Each eval row is one question; rows sharing an identical state are batched into one request.
def build_requests(rows):
    groups = collections.OrderedDict()
    for r in rows:
        key = (r["dataset"], json.dumps(r["state"], sort_keys=True, ensure_ascii=False)) if BATCH_SAME_STATE \
            else (r["dataset"], r["id"])
        groups.setdefault(key, []).append(r)
    reqs = []
    for members in groups.values():
        qmap = {f"q{i}": m["id"] for i, m in enumerate(members)}
        questions = {f"q{i}": m["questions"][m["question_id"]] for i, m in enumerate(members)}
        rid = hashlib.sha1("|".join(sorted(qmap.values())).encode()).hexdigest()[:16]
        reqs.append({"request_id": rid, "dataset": members[0]["dataset"], "qmap": qmap,
                     "body": {"state": members[0]["state"], "questions": questions}})
    return reqs


REQUESTS = build_requests(rows)
print(f"{len(rows)} rows -> {len(REQUESTS)} requests "
      f"(max {max(len(r['qmap']) for r in REQUESTS)} questions in one request)")

7287 rows -> 5962 requests (max 9 questions in one request)


In [5]:
# Mock responder (only used when MOCK=True): mimics the API response schema with synthetic probabilities.
GOLD_BY_ID = {r["id"]: r["gold"] for r in rows}


def _mock_answer(q, gold, seed):
    rng = random.Random(seed)
    t = q["type"]
    if t == "noul":
        p = rng.betavariate(6, 2) if rng.random() < 0.8 else rng.betavariate(2, 6)
        return {"type": "noul", "noul": round(p if gold else 1 - p, 4)}
    keys = list(q["criteria"]) if t == "choice" else [str(i) for i in range(len(q["criteria"]))]
    g = str(gold)
    w = [rng.gammavariate(0.6, 1) for _ in keys]
    if rng.random() < 0.75:
        w[keys.index(g)] += rng.uniform(1, 6)
    s = sum(w)
    probs = {k: x / s for k, x in zip(keys, w)}
    pm = max(probs.values())
    conf = (len(keys) * pm - 1) / (len(keys) - 1)
    top = max(probs, key=probs.get)
    if t == "choice":
        return {"type": "choice", "choice": top, "probabilities": probs, "confidence": conf}
    return {"type": "score", "score": sum(int(k) * v for k, v in probs.items()),
            "legend": {str(i): str(c)[:40] for i, c in enumerate(q["criteria"])},
            "probabilities": probs, "confidence": conf}


def mock_call(req):
    ans = {}
    for qid, q in req["body"]["questions"].items():
        rid = req["qmap"][qid]
        ans[qid] = _mock_answer(q, GOLD_BY_ID[rid], int(hashlib.sha1(rid.encode()).hexdigest()[:8], 16))
    n_tok = len(json.dumps(req["body"])) // 4
    return {"status": 200, "latency_s": round(random.uniform(0.15, 0.6), 4), "attempts": 1,
            "response": {"model": "MOCK", "answers": ans, "usage": {"input_tokens": n_tok, "output_tokens": 0}}}

In [6]:
# HTTP client (stdlib only) with retries and exponential backoff + jitter.
ENDPOINT = f"{BASE_URL}/v1/systemone"
RETRY_STATUS = {429, 500, 502, 503, 504, 529}


def call_api(req):
    if MOCK:
        return mock_call(req)
    data = json.dumps(dict(req["body"], model=MODEL), ensure_ascii=False).encode("utf-8")
    last = None
    for attempt in range(1, MAX_ATTEMPTS + 1):
        t0 = time.time()
        http = urllib.request.Request(ENDPOINT, data=data, method="POST", headers={
            "Authorization": f"Bearer {API_KEY}", "Content-Type": "application/json"})
        try:
            with urllib.request.urlopen(http, timeout=TIMEOUT_S) as resp:
                return {"status": resp.status, "latency_s": round(time.time() - t0, 4), "attempts": attempt,
                        "response": json.loads(resp.read().decode("utf-8"))}
        except urllib.error.HTTPError as e:
            last = {"status": e.code, "latency_s": round(time.time() - t0, 4), "attempts": attempt,
                    "error": e.read().decode("utf-8", "replace")[:2000]}
            if e.code not in RETRY_STATUS:
                return last
        except Exception as e:  # DNS, TLS, timeout, connection reset
            last = {"status": None, "latency_s": round(time.time() - t0, 4), "attempts": attempt,
                    "error": f"{type(e).__name__}: {e}"}
        time.sleep(min(60, 2 ** attempt) * (0.5 + random.random()))
    return last


# Preflight: fail fast with a clear message instead of retrying thousands of requests.
if not MOCK:
    host = urllib.parse.urlparse(BASE_URL).hostname
    try:
        socket.getaddrinfo(host, 443)
    except socket.gaierror as e:
        raise SystemExit(f"Cannot resolve {host} ({e}). This machine's DNS/network blocks the Jev API; "
                         "run the notebook where it is reachable or set TYPESAFE_BASE_URL / HTTPS_PROXY.")
    probe = call_api({"body": {"state": "Help! My payouts have been failing for 3 days.",
                               "questions": {"q0": {"type": "noul", "instructions": "Does this convey urgency?"}}}})
    if probe.get("status") != 200:
        raise SystemExit(f"Preflight request failed: status={probe.get('status')} {probe.get('error', '')[:300]}")
    print("preflight OK:", probe["response"].get("model"), probe["response"]["answers"]["q0"],
          f"{probe['latency_s']}s")

preflight OK: jev-1.13.0 {'type': 'noul', 'noul': 0.95} 0.2029s


In [7]:
# Run the eval (resumable). Raw responses are appended to RESPONSES_PATH as JSON lines.
RESPONSES_PATH = Path(OUT_DIR, "responses_MOCK.jsonl" if MOCK else "responses.jsonl")


def load_done():
    done = {}
    if RESPONSES_PATH.exists():
        for l in open(RESPONSES_PATH, encoding="utf-8"):
            rec = json.loads(l)
            if rec.get("status") == 200:
                done[rec["request_id"]] = rec
    return done


done = load_done()
pending = [r for r in REQUESTS if r["request_id"] not in done]
if LIMIT_REQUESTS:
    pending = pending[:LIMIT_REQUESTS]
print(f"{len(done)} requests already answered; sending {len(pending)} with {WORKERS} workers")

lock, stats, t_start = threading.Lock(), collections.Counter(), time.time()
with open(RESPONSES_PATH, "a", encoding="utf-8") as out, cf.ThreadPoolExecutor(WORKERS) as pool:
    futs = {pool.submit(call_api, r): r for r in pending}
    for n, fut in enumerate(cf.as_completed(futs), 1):
        r = futs[fut]
        res = fut.result()
        rec = {"request_id": r["request_id"], "dataset": r["dataset"], "qmap": r["qmap"], "model_requested": MODEL,
               "ts": time.time(), **res}
        with lock:
            out.write(json.dumps(rec, ensure_ascii=False) + "\n")
            out.flush()
            stats[res.get("status")] += 1
        if n % 250 == 0 or n == len(pending):
            print(f"  {n}/{len(pending)}  {time.time() - t_start:.0f}s  statuses={dict(stats)}", flush=True)

done = load_done()
print(f"answered requests: {len(done)}/{len(REQUESTS)}")

0 requests already answered; sending 5962 with 8 workers
  250/5962  6s  statuses={200: 250}
  500/5962  12s  statuses={200: 500}
  750/5962  19s  statuses={200: 750}
  1000/5962  25s  statuses={200: 1000}
  1250/5962  30s  statuses={200: 1250}
  1500/5962  37s  statuses={200: 1500}
  1750/5962  43s  statuses={200: 1750}
  2000/5962  49s  statuses={200: 2000}
  2250/5962  55s  statuses={200: 2250}
  2500/5962  61s  statuses={200: 2500}
  2750/5962  67s  statuses={200: 2750}
  3000/5962  73s  statuses={200: 3000}
  3250/5962  79s  statuses={200: 3250}
  3500/5962  85s  statuses={200: 3500}
  3750/5962  91s  statuses={200: 3750}
  4000/5962  97s  statuses={200: 4000}
  4250/5962  103s  statuses={200: 4250}
  4500/5962  109s  statuses={200: 4500}
  4750/5962  115s  statuses={200: 4750}
  5000/5962  122s  statuses={200: 5000}
  5250/5962  128s  statuses={200: 5250}
  5500/5962  135s  statuses={200: 5500}
  5750/5962  143s  statuses={200: 5750}
  5962/5962  149s  statuses={200: 5962}
answer

In [8]:
# Parse every answer into one prediction row per eval row.
EPS = 1e-12
row_by_id = {r["id"]: r for r in rows}
all_attempts = [json.loads(l) for l in open(RESPONSES_PATH, encoding="utf-8")] if RESPONSES_PATH.exists() else []


def option_keys(q):
    return list(q["criteria"]) if q["type"] == "choice" else (
        [str(i) for i in range(len(q["criteria"]))] if q["type"] == "score" else ["true", "false"])


def parse_answer(r, a):
    q = r["questions"][r["question_id"]]
    t, keys, gold = q["type"], option_keys(q), r["gold"]
    rec = {"id": r["id"], "dataset": r["dataset"], "subset": r.get("subset"), "primitive": t, "K": len(keys),
           "gold": json.dumps(gold), "answered": a is not None}
    if a is None:
        return rec
    if t == "noul":
        p_true = float(a["noul"])
        probs = {"true": p_true, "false": 1.0 - p_true}
        pred_key = "true" if p_true >= 0.5 else "false"
        gold_key = "true" if gold else "false"
        rec.update(api_confidence=np.nan, noul=p_true, expected_score=np.nan)
    else:
        probs = {str(k): float(v) for k, v in a.get("probabilities", {}).items()}
        missing = [k for k in keys if k not in probs]
        for k in missing:
            probs[k] = 0.0
        pred_key = str(a["choice"]) if t == "choice" else max(keys, key=lambda k: probs[k])
        gold_key = str(gold)
        rec.update(api_confidence=float(a.get("confidence", np.nan)), noul=np.nan,
                   expected_score=float(a["score"]) if t == "score" else np.nan, missing_option_probs=len(missing))
    s = sum(probs.values())
    vec = np.array([probs[k] for k in keys], dtype=float) / (s if s > 0 else 1.0)
    p_max = float(vec.max())
    rec.update(
        pred=pred_key, correct=bool(pred_key == gold_key), prob_sum=s,
        p_gold=float(vec[keys.index(gold_key)]), p_max=p_max,
        conf_formula=(p_max - 1 / len(keys)) / (1 - 1 / len(keys)),          # article: (p_max - 1/K)/(1 - 1/K)
        brier_sum=float(((vec - np.eye(len(keys))[keys.index(gold_key)]) ** 2).sum()),
        probs=json.dumps(dict(zip(keys, vec.round(6).tolist()))),
    )
    if t == "score":
        cdf_p, cdf_y = np.cumsum(vec), (np.arange(len(keys)) >= int(gold)).astype(float)
        rec.update(abs_err_expected=abs(rec["expected_score"] - int(gold)),
                   within1=abs(int(pred_key) - int(gold)) <= 1,
                   rps=float(((cdf_p - cdf_y)[:-1] ** 2).sum() / (len(keys) - 1)))
    return rec


pred_rows, answered_ids = [], set()
for rec in done.values():
    for qid, rid in rec["qmap"].items():
        a = rec["response"].get("answers", {}).get(qid)
        if a is not None and rid in row_by_id:
            pred_rows.append(parse_answer(row_by_id[rid], a))
            answered_ids.add(rid)
for rid, r in row_by_id.items():
    if rid not in answered_ids:
        pred_rows.append(parse_answer(r, None))
pred = pd.DataFrame(pred_rows)
ans = pred[pred.answered].copy()
print(f"answered rows: {len(ans)}/{len(pred)}  |  prob-sum range: "
      f"{ans.prob_sum.min():.4f}–{ans.prob_sum.max():.4f}  |  rows with missing option probs: "
      f"{int(ans.get('missing_option_probs', pd.Series(dtype=float)).fillna(0).gt(0).sum())}")

answered rows: 7287/7287  |  prob-sum range: 0.9900–1.0000  |  rows with missing option probs: 0


## Metric definitions
* **accuracy** – argmax / `choice` / `noul ≥ 0.5` equals gold; Score uses the most probable level. 95% Wilson CI.
* **majority baseline / balanced accuracy / macro-F1** – computed inside each label space (dataset; BBH task)
  and n-weighted when a table row spans several datasets, so unrelated label sets are never pooled.
* **NLL (log loss)** – mean −ln p(gold): the logarithmic proper scoring rule RLCD optimises (lower is better; uniform = ln K).
* **Brier** – Σₖ(pₖ − yₖ)² per item (0 = perfect, 2 = worst). **Brier (article, 1/K)** – the same divided by K.
* **ECE** – Σ_b |B_b|/N · |acc(B_b) − conf(B_b)| over 10 equal-width bins of top-label confidence p_max;
  **ECE (equal-mass)** uses 10 equal-count bins; **MCE** = max bin gap; **overconfidence** = mean p_max − accuracy.
* **AUROC (conf→correct)** – does higher confidence separate right from wrong answers (0.5 = no signal).
* **AURC** – area under the risk–coverage curve when answering the most confident items first (lower is better).
* **τ routing** – auto-route when p_max ≥ τ = (C_err − C_human)/(C_err − C_auto); expected cost per decision uses
  the realised correctness: auto & correct → C_auto, auto & wrong → C_err, deferred → C_human.

In [9]:
def wilson(k, n, z=1.96):
    if n == 0:
        return (np.nan, np.nan)
    p = k / n
    d = 1 + z * z / n
    c = (p + z * z / (2 * n)) / d
    h = z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / d
    return (c - h, c + h)


def auroc(scores, labels):
    """Mann–Whitney AUROC with average ranks for ties; nan if one class is absent."""
    s, y = np.asarray(scores, float), np.asarray(labels, bool)
    npos, nneg = y.sum(), (~y).sum()
    if npos == 0 or nneg == 0:
        return np.nan
    ranks = pd.Series(s).rank(method="average").to_numpy()
    return float((ranks[y].sum() - npos * (npos + 1) / 2) / (npos * nneg))


def auprc(scores, labels):
    s, y = np.asarray(scores, float), np.asarray(labels, bool)
    if y.sum() == 0:
        return np.nan
    order = np.argsort(-s, kind="mergesort")
    y = y[order]
    tp = np.cumsum(y)
    precision = tp / np.arange(1, len(y) + 1)
    return float((precision * y).sum() / y.sum())          # average precision


def ece(conf, correct, bins=ECE_BINS, equal_mass=False):
    conf, correct = np.asarray(conf, float), np.asarray(correct, float)
    if len(conf) == 0:
        return np.nan, np.nan, []
    if equal_mass:
        edges = np.unique(np.quantile(conf, np.linspace(0, 1, bins + 1)))
        idx = np.clip(np.searchsorted(edges, conf, side="right") - 1, 0, len(edges) - 2)
    else:
        edges = np.linspace(0, 1, bins + 1)
        idx = np.clip(np.digitize(conf, edges[1:-1], right=False), 0, bins - 1)
    total, worst, table = 0.0, 0.0, []
    for b in range(len(edges) - 1):
        m = idx == b
        if m.sum() == 0:
            continue
        gap = abs(correct[m].mean() - conf[m].mean())
        total += m.sum() / len(conf) * gap
        worst = max(worst, gap)
        table.append({"bin_lo": edges[b], "bin_hi": edges[b + 1], "n": int(m.sum()),
                      "mean_conf": conf[m].mean(), "accuracy": correct[m].mean(), "gap": gap})
    return float(total), float(worst), table


def risk_coverage(conf, correct):
    order = np.argsort(-np.asarray(conf, float), kind="mergesort")
    c = np.asarray(correct, float)[order]
    n = np.arange(1, len(c) + 1)
    risk = np.cumsum(1 - c) / n
    return n / len(c), risk


TAU = (C_ERROR - C_HUMAN) / (C_ERROR - C_AUTO)


def routing(conf, correct, tau):
    conf, correct = np.asarray(conf, float), np.asarray(correct, bool)
    auto = conf >= tau
    cost = np.where(auto, np.where(correct, C_AUTO, C_ERROR), C_HUMAN)
    all_auto = np.where(correct, C_AUTO, C_ERROR).mean()
    return {"coverage": auto.mean(), "acc_auto": correct[auto].mean() if auto.any() else np.nan,
            "cost_per_decision": cost.mean(), "cost_all_auto": all_auto, "cost_all_human": C_HUMAN,
            "saving_vs_all_human": 1 - cost.mean() / C_HUMAN}


def _gold_key(g):
    v = json.loads(g)
    return ("true" if v else "false") if isinstance(v, bool) else str(v)


def class_metrics(df):
    """Majority baseline, balanced accuracy and macro-F1 within ONE label space (one dataset / BBH task)."""
    gold = df.gold.map(_gold_key)
    predk = df.pred.astype(str)
    counts = gold.value_counts()
    recalls, f1s = [], []
    for g in counts.index:
        is_g, pred_g = gold == g, predk == g
        tp = (is_g & pred_g).sum()
        recalls.append(tp / is_g.sum())
        prec = tp / pred_g.sum() if pred_g.sum() else 0.0
        f1s.append(0.0 if prec + recalls[-1] == 0 else 2 * prec * recalls[-1] / (prec + recalls[-1]))
    return counts.iloc[0] / len(df), float(np.mean(recalls)), float(np.mean(f1s))


def unit_of(df):
    return np.where(df.dataset == "BBH", "BBH/" + df.subset.fillna(""), df.dataset)


def metrics(df):
    n = len(df)
    if n == 0:
        return {}
    k = int(df.correct.sum())
    lo, hi = wilson(k, n)
    # label-space metrics are computed per dataset (per task for BBH) and n-weighted across units
    parts = [(len(g), class_metrics(g)) for _, g in df.groupby(unit_of(df))]
    maj, bal, mf1 = (sum(w * m[i] for w, m in parts) / n for i in range(3))
    e10, mce, _ = ece(df.p_max, df.correct)
    em, _, _ = ece(df.p_max, df.correct, equal_mass=True)
    cov, risk = risk_coverage(df.p_max, df.correct)
    out = {
        "n": n, "accuracy": k / n, "acc_ci_lo": lo, "acc_ci_hi": hi,
        "balanced_accuracy": bal, "macro_f1": mf1, "majority_baseline": maj,
        "nll": float(-np.log(np.clip(df.p_gold, EPS, 1)).mean()),
        "nll_uniform": float(np.log(df.K).mean()),
        "brier": float(df.brier_sum.mean()), "brier_article_1_over_K": float((df.brier_sum / df.K).mean()),
        "ece": e10, "ece_equal_mass": em, "mce": mce,
        "mean_conf": float(df.p_max.mean()), "overconfidence": float(df.p_max.mean() - k / n),
        "auroc_conf_correct": auroc(df.p_max, df.correct), "aurc": float(np.mean(risk)),
    }
    if df.api_confidence.notna().any():
        out["auroc_api_confidence"] = auroc(df.api_confidence.fillna(df.conf_formula), df.correct)
    r = routing(df.p_max, df.correct, TAU)
    out.update({f"tau_{k_}": v for k_, v in r.items()})
    for t in CONF_THRESHOLDS:
        m = df.p_max >= t
        out[f"coverage@{t}"] = float(m.mean())
        out[f"acc@{t}"] = float(df.correct[m].mean()) if m.any() else np.nan
    return out


def table(groupby):
    recs = []
    for key, g in ans.groupby(groupby, dropna=False):
        key = key if isinstance(key, tuple) else (key,)
        recs.append({**dict(zip(groupby if isinstance(groupby, list) else [groupby], key)), **metrics(g)})
    return pd.DataFrame(recs)


# --- self-test of the metric code on synthetic data with known answers ---
_t = pd.DataFrame({"correct": [True] * 8 + [False] * 2, "p_max": [0.8] * 10, "p_gold": [0.8] * 8 + [0.1] * 2,
                   "K": [2] * 10, "brier_sum": [0.08] * 8 + [1.28] * 2, "gold": ['"A"'] * 10,
                   "pred": ["A"] * 8 + ["B"] * 2, "api_confidence": [np.nan] * 10, "conf_formula": [0.6] * 10,
                   "dataset": ["X"] * 10, "subset": [None] * 10})
_m = metrics(_t)
assert abs(_m["accuracy"] - 0.8) < 1e-9 and abs(_m["ece"]) < 1e-9 and abs(_m["overconfidence"]) < 1e-9
assert abs(_m["majority_baseline"] - 1.0) < 1e-9 and abs(_m["balanced_accuracy"] - 0.8) < 1e-9
assert abs(auroc([0.9, 0.8, 0.3, 0.2], [1, 1, 0, 0]) - 1.0) < 1e-9 and abs(auroc([0.5] * 4, [1, 0, 1, 0]) - 0.5) < 1e-9
assert abs(TAU - 4 / 4.95) < 1e-12
print("metric self-tests passed; tau =", round(TAU, 4))

metric self-tests passed; tau = 0.8081


In [10]:
overall = pd.DataFrame([{"scope": "ALL", **metrics(ans)}])
by_primitive = table("primitive")
by_dataset = table("dataset").sort_values("dataset")
by_dataset_primitive = table(["dataset", "primitive"])
by_bbh_task = table(["dataset", "subset"]).query("dataset == 'BBH'").drop(columns="dataset")
by_subset = table(["dataset", "subset"]).query("dataset in ['RAGTruth', 'JevBench-Hard']")

# answered coverage per dataset (rows with an API error / no answer are excluded from the metrics above)
cover = pred.groupby("dataset").answered.agg(["size", "sum"]).rename(columns={"size": "eval_rows", "sum": "answered"})
by_dataset = by_dataset.merge(cover, left_on="dataset", right_index=True)

KEY_COLS = ["n", "accuracy", "acc_ci_lo", "acc_ci_hi", "majority_baseline", "balanced_accuracy", "macro_f1", "nll",
            "nll_uniform", "brier", "ece", "mce", "overconfidence", "auroc_conf_correct", "aurc",
            "tau_coverage", "tau_acc_auto", "tau_cost_per_decision"]
fmt = lambda d: d.round(3)  # plain rounding: no jinja2 / Styler dependency
print("OVERALL"); display(fmt(overall[["scope"] + KEY_COLS]))
print("PER DATASET"); display(fmt(by_dataset[["dataset", "eval_rows", "answered"] + KEY_COLS]))
print("PER PRIMITIVE"); display(fmt(by_primitive[["primitive"] + KEY_COLS]))

OVERALL


,scope,n,accuracy,acc_ci_lo,acc_ci_hi,majority_baseline,balanced_accuracy,macro_f1,nll,nll_uniform,brier,ece,mce,overconfidence,auroc_conf_correct,aurc,tau_coverage,tau_acc_auto,tau_cost_per_decision
0,ALL,7287,0.873,0.866,0.881,0.535,0.883,0.854,0.392,0.966,0.183,0.02,0.097,0.019,0.837,0.035,0.782,0.943,0.477


PER DATASET


,dataset,eval_rows,answered,n,accuracy,acc_ci_lo,acc_ci_hi,majority_baseline,balanced_accuracy,macro_f1,nll,nll_uniform,brier,ece,mce,overconfidence,auroc_conf_correct,aurc,tau_coverage,tau_acc_auto,tau_cost_per_decision
0,BBH,700,700,700,0.900,0.876,0.920,0.413,0.906,0.893,0.267,1.232,0.132,0.034,0.360,-0.002,0.909,0.015,0.816,0.975,0.324
1,Belebele,500,500,500,0.978,0.961,0.988,0.276,0.978,0.978,0.257,1.386,0.034,0.010,0.307,0.009,0.934,0.004,0.976,0.988,0.132
2,CIRCA,500,500,500,0.902,0.873,0.925,0.474,0.805,0.751,0.295,1.609,0.150,0.032,0.300,-0.028,0.877,0.020,0.750,0.976,0.377
3,ContractNLI,500,500,500,0.838,0.803,0.868,0.456,0.849,0.820,1.038,1.099,0.256,0.067,0.630,0.055,0.769,0.065,0.768,0.911,0.607
4,FinancialPhraseBank,1000,1000,1000,0.921,0.903,0.936,0.598,0.950,0.921,0.194,1.099,0.118,0.017,0.457,0.010,0.896,0.012,0.861,0.965,0.331
5,JevBench-Hard,293,293,293,0.672,0.617,0.724,0.232,0.695,0.685,0.755,1.362,0.426,0.096,0.240,0.093,0.729,0.172,0.474,0.856,0.887
6,JudgeBench,300,300,300,0.790,0.740,0.832,0.563,0.793,0.789,0.450,0.693,0.288,0.034,0.060,-0.003,0.752,0.097,0.493,0.919,0.729
7,RAGTruth,1494,1494,1494,0.777,0.755,0.797,0.778,0.823,0.741,0.504,0.693,0.322,0.043,0.062,0.043,0.690,0.134,0.626,0.856,0.853
8,TabFact,500,500,500,0.932,0.906,0.951,0.510,0.932,0.932,0.180,0.693,0.102,0.015,0.069,-0.012,0.880,0.013,0.870,0.972,0.292
9,TruthfulQA-Binary,500,500,500,0.964,0.944,0.977,0.508,0.964,0.964,0.197,0.693,0.056,0.018,0.446,0.008,0.866,0.008,0.954,0.981,0.183


PER PRIMITIVE


,primitive,n,accuracy,acc_ci_lo,acc_ci_hi,majority_baseline,balanced_accuracy,macro_f1,nll,nll_uniform,brier,ece,mce,overconfidence,auroc_conf_correct,aurc,tau_coverage,tau_acc_auto,tau_cost_per_decision
0,choice,3844,0.900,0.890,0.909,0.437,0.890,0.877,0.426,1.089,0.149,0.019,0.300,0.013,0.851,0.025,0.825,0.958,0.388
1,noul,2372,0.818,0.802,0.833,0.683,0.848,0.795,0.407,0.693,0.257,0.028,0.054,0.028,0.770,0.074,0.690,0.905,0.670
2,score,1071,0.901,0.882,0.918,0.587,0.936,0.900,0.233,1.127,0.143,0.023,0.340,0.018,0.899,0.016,0.832,0.961,0.371


In [11]:
# Primitive-specific metrics
score_df = ans[ans.primitive == "score"]
noul_df = ans[ans.primitive == "noul"]
score_metrics = (score_df.groupby("dataset").agg(n=("id", "size"), exact_acc=("correct", "mean"),
                                                 within1_acc=("within1", "mean"), mae_expected=("abs_err_expected", "mean"),
                                                 rps=("rps", "mean")).reset_index() if len(score_df) else pd.DataFrame())
noul_recs = []
for ds, g in noul_df.groupby(["dataset", "subset"], dropna=False):
    y = g.gold.map(json.loads).astype(bool)
    p = g.noul
    yhat = p >= 0.5
    tp, fp, fn = int((yhat & y).sum()), int((yhat & ~y).sum()), int((~yhat & y).sum())
    noul_recs.append({"dataset": ds[0], "subset": ds[1], "n": len(g), "base_rate_true": y.mean(),
                      "accuracy": (yhat == y).mean(), "auroc": auroc(p, y), "auprc": auprc(p, y),
                      "precision_true": tp / (tp + fp) if tp + fp else np.nan,
                      "recall_true": tp / (tp + fn) if tp + fn else np.nan,
                      "f1_true": 2 * tp / (2 * tp + fp + fn) if tp else 0.0,
                      "brier_binary": float(((p - y) ** 2).mean())})
noul_metrics = pd.DataFrame(noul_recs)
print("SCORE primitive"); display(fmt(score_metrics))
print("NOUL primitive"); display(fmt(noul_metrics))
print("BBH per task"); display(fmt(by_bbh_task[["subset"] + KEY_COLS]))

SCORE primitive


,dataset,n,exact_acc,within1_acc,mae_expected,rps
0,FinancialPhraseBank,1000,0.921,0.999,0.108,0.030
1,JevBench-Hard,71,0.620,0.971831,0.420,0.088


NOUL primitive


,dataset,subset,n,base_rate_true,accuracy,auroc,auprc,precision_true,recall_true,f1_true,brier_binary
0,BBH,boolean_expressions,50,0.540,1.000,1.000,1.000,1.000,1.000,1.000,0.001
1,BBH,causal_judgement,50,0.500,0.540,0.713,0.750,0.528,0.760,0.623,0.244
2,BBH,formal_fallacies,50,0.520,1.000,1.000,1.000,1.000,1.000,1.000,0.005
3,BBH,navigate,50,0.360,1.000,1.000,1.000,1.000,1.000,1.000,0.019
4,BBH,web_of_lies,50,0.480,1.000,1.000,1.000,1.000,1.000,1.000,0.006
5,JevBench-Hard,ads_media,2,0.500,1.000,1.000,1.000,1.000,1.000,1.000,0.021
6,JevBench-Hard,agent_action_approval,3,0.333,0.667,1.000,1.000,NaN,0.000,0.000,0.182
7,JevBench-Hard,agent_memory,1,1.000,1.000,NaN,1.000,1.000,1.000,1.000,0.230
8,JevBench-Hard,agent_routing,7,0.429,0.429,0.625,0.556,0.400,0.667,0.500,0.306
9,JevBench-Hard,authorization_reconciliation,3,0.333,0.667,1.000,1.000,0.500,1.000,0.667,0.161


BBH per task


,subset,n,accuracy,acc_ci_lo,acc_ci_hi,majority_baseline,balanced_accuracy,macro_f1,nll,nll_uniform,brier,ece,mce,overconfidence,auroc_conf_correct,aurc,tau_coverage,tau_acc_auto,tau_cost_per_decision
0,boolean_expressions,50,1.00,0.929,1.000,0.54,1.000,1.000,0.032,0.693,0.003,0.031,0.031,-0.031,NaN,0.000,1.00,1.000,0.050
1,causal_judgement,50,0.54,0.404,0.670,0.50,0.540,0.517,0.677,0.693,0.488,0.197,0.345,0.197,0.778,0.249,0.34,0.824,0.974
2,date_understanding,50,0.90,0.786,0.957,0.26,0.906,0.902,0.322,1.766,0.153,0.113,0.320,-0.041,0.793,0.035,0.74,0.946,0.495
3,disambiguation_qa,50,0.60,0.462,0.724,0.76,0.712,0.559,0.762,1.116,0.510,0.174,0.609,0.168,0.650,0.256,0.48,0.792,1.039
4,formal_fallacies,50,1.00,0.929,1.000,0.52,1.000,1.000,0.065,0.693,0.010,0.062,0.147,-0.062,NaN,0.000,1.00,1.000,0.050
5,geometric_shapes,50,1.00,0.929,1.000,0.20,1.000,1.000,0.114,2.303,0.038,0.098,0.520,-0.098,NaN,0.000,0.86,1.000,0.183
6,logical_deduction_five_objects,50,0.96,0.865,0.989,0.24,0.946,0.955,0.107,1.609,0.058,0.045,0.690,-0.011,0.995,0.001,0.92,1.000,0.126
7,logical_deduction_three_objects,50,1.00,0.929,1.000,0.34,1.000,1.000,0.001,1.099,0.000,0.001,0.001,-0.001,NaN,0.000,1.00,1.000,0.050
8,navigate,50,1.00,0.929,1.000,0.64,1.000,1.000,0.098,0.693,0.038,0.085,0.475,-0.085,NaN,0.000,0.88,1.000,0.164
9,salient_translation_error_detection,50,0.78,0.648,0.872,0.26,0.764,0.752,1.054,1.792,0.309,0.130,0.540,0.055,0.800,0.079,0.66,0.879,0.769


In [12]:
# Threshold sweep (selective prediction) and cost-based routing per dataset
sweep = []
for t in np.round(np.arange(0.0, 1.0001, 0.05), 2):
    m = ans.p_max >= t
    sweep.append({"threshold": t, "coverage": m.mean(), "accuracy": ans.correct[m].mean() if m.any() else np.nan,
                  **{k: v for k, v in routing(ans.p_max, ans.correct, t).items() if k == "cost_per_decision"}})
sweep = pd.DataFrame(sweep)
routing_by_dataset = by_dataset[["dataset", "n", "accuracy", "tau_coverage", "tau_acc_auto", "tau_cost_per_decision",
                                 "tau_cost_all_auto", "tau_saving_vs_all_human"]]
print(f"tau = (C_err - C_human)/(C_err - C_auto) = ({C_ERROR}-{C_HUMAN})/({C_ERROR}-{C_AUTO}) = {TAU:.4f}")
display(fmt(routing_by_dataset))
display(fmt(sweep))

tau = (C_err - C_human)/(C_err - C_auto) = (5.0-1.0)/(5.0-0.05) = 0.8081


,dataset,n,accuracy,tau_coverage,tau_acc_auto,tau_cost_per_decision,tau_cost_all_auto,tau_saving_vs_all_human
0,BBH,700,0.900,0.816,0.975,0.324,0.545,0.676
1,Belebele,500,0.978,0.976,0.988,0.132,0.159,0.868
2,CIRCA,500,0.902,0.750,0.976,0.377,0.535,0.623
3,ContractNLI,500,0.838,0.768,0.911,0.607,0.852,0.393
4,FinancialPhraseBank,1000,0.921,0.861,0.965,0.331,0.441,0.669
5,JevBench-Hard,293,0.672,0.474,0.856,0.887,1.672,0.113
6,JudgeBench,300,0.790,0.493,0.919,0.729,1.090,0.271
7,RAGTruth,1494,0.777,0.626,0.856,0.853,1.153,0.147
8,TabFact,500,0.932,0.870,0.972,0.292,0.387,0.708
9,TruthfulQA-Binary,500,0.964,0.954,0.981,0.183,0.228,0.817


,threshold,coverage,accuracy,cost_per_decision
0,0.00,1.000,0.873,0.676
1,0.05,1.000,0.873,0.676
2,0.10,1.000,0.873,0.676
3,0.15,1.000,0.873,0.676
4,0.20,1.000,0.873,0.676
5,0.25,1.000,0.873,0.676
6,0.30,1.000,0.874,0.676
7,0.35,0.999,0.874,0.674
8,0.40,0.998,0.874,0.672
9,0.45,0.996,0.875,0.670


In [13]:
# Figures: reliability diagrams + risk-coverage
FIG = Path(REPORT_DIR, "figures")


def reliability(ax, df, title):
    _, _, tab = ece(df.p_max, df.correct)
    e, _, _ = ece(df.p_max, df.correct)
    ax.plot([0, 1], [0, 1], "--", color="grey", lw=1)
    if tab:
        t = pd.DataFrame(tab)
        w = (t.bin_hi - t.bin_lo) * 0.9
        ax.bar(t.bin_lo, t.accuracy, width=w, align="edge", alpha=0.7, edgecolor="k", label="accuracy")
        ax.plot(t.mean_conf, t.accuracy, "o-", color="C3", ms=3, label="(conf, acc)")
    ax.set(xlim=(0, 1), ylim=(0, 1), title=f"{title}\nn={len(df)} acc={df.correct.mean():.3f} ECE={e:.3f}",
           xlabel="confidence p_max", ylabel="accuracy")
    ax.title.set_fontsize(9)


fig, axes = plt.subplots(1, 4, figsize=(18, 4.5))
reliability(axes[0], ans, "ALL")
for ax, p in zip(axes[1:], ["choice", "score", "noul"]):
    reliability(ax, ans[ans.primitive == p], p)
fig.tight_layout(); fig.savefig(FIG / "reliability_overall.png", dpi=110); plt.close(fig)

dsets = sorted(ans.dataset.unique())
cols = 4
fig, axes = plt.subplots(math.ceil(len(dsets) / cols), cols, figsize=(18, 4.2 * math.ceil(len(dsets) / cols)))
for ax, d in zip(axes.ravel(), dsets):
    reliability(ax, ans[ans.dataset == d], d)
for ax in axes.ravel()[len(dsets):]:
    ax.axis("off")
fig.tight_layout(); fig.savefig(FIG / "reliability_by_dataset.png", dpi=100); plt.close(fig)

fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))
for d in dsets:
    g = ans[ans.dataset == d]
    c, r = risk_coverage(g.p_max, g.correct)
    ax[0].plot(c, 1 - r, lw=1, label=d)
c, r = risk_coverage(ans.p_max, ans.correct)
ax[0].plot(c, 1 - r, "k", lw=2.5, label="ALL")
ax[0].set(xlabel="coverage (most confident first)", ylabel="accuracy of answered", title="Accuracy–coverage")
ax[0].legend(fontsize=7, ncol=2)
ax[1].bar(by_dataset.dataset, by_dataset.accuracy, yerr=[by_dataset.accuracy - by_dataset.acc_ci_lo,
                                                          by_dataset.acc_ci_hi - by_dataset.accuracy], capsize=3)
ax[1].scatter(by_dataset.dataset, by_dataset.majority_baseline, color="C3", zorder=3, label="majority baseline")
ax[1].set(ylim=(0, 1), title="Accuracy per dataset (95% CI)"); ax[1].legend(fontsize=8)
ax[1].tick_params(axis="x", rotation=60, labelsize=8)
fig.tight_layout(); fig.savefig(FIG / "accuracy_coverage.png", dpi=110); plt.close(fig)
print("figures:", sorted(p.name for p in FIG.iterdir()))

figures: ['accuracy_coverage.png', 'reliability_by_dataset.png', 'reliability_overall.png']


In [14]:
# Operations: requests, errors, latency, tokens, cost
ok = [r for r in all_attempts if r.get("status") == 200]
lat = np.array([r["latency_s"] for r in ok]) if ok else np.array([np.nan])
in_tok = sum((r["response"].get("usage") or {}).get("input_tokens", 0) for r in done.values())
out_tok = sum((r["response"].get("usage") or {}).get("output_tokens", 0) for r in done.values())
models = collections.Counter(r["response"].get("model") for r in done.values())
err_status = collections.Counter(r.get("status") for r in all_attempts if r.get("status") != 200)
ops = {
    "mock": MOCK, "model_requested": MODEL, "models_reported": dict(models), "endpoint": ENDPOINT,
    "eval_rows": len(rows), "requests_total": len(REQUESTS), "requests_answered": len(done),
    "rows_answered": int(pred.answered.sum()), "requests_unanswered": len(REQUESTS) - len(done),
    "failed_attempt_statuses_all_runs": {str(k): v for k, v in err_status.items()},
    "latency_p50_s": float(np.nanpercentile(lat, 50)), "latency_p95_s": float(np.nanpercentile(lat, 95)),
    "latency_p99_s": float(np.nanpercentile(lat, 99)), "latency_mean_s": float(np.nanmean(lat)),
    "input_tokens": int(in_tok), "output_tokens": int(out_tok),
    "cost_usd": in_tok / 1e6 * PRICE_PER_M_INPUT, "price_per_m_input_usd": PRICE_PER_M_INPUT,
}
print(json.dumps(ops, indent=1))

{
 "mock": false,
 "model_requested": "jev-latest",
 "models_reported": {
  "jev-1.13.0": 5962
 },
 "endpoint": "https://api.typesafe.ai/v1/systemone",
 "eval_rows": 7287,
 "requests_total": 5962,
 "requests_answered": 5962,
 "rows_answered": 7287,
 "requests_unanswered": 0,
 "failed_attempt_statuses_all_runs": {},
 "latency_p50_s": 0.1918,
 "latency_p95_s": 0.2616,
 "latency_p99_s": 0.3164,
 "latency_mean_s": 0.19890345521637035,
 "input_tokens": 4383856,
 "output_tokens": 242117,
 "cost_usd": 0.17535424,
 "price_per_m_input_usd": 0.04
}


In [15]:
# Write the report (Markdown + self-contained HTML), tables (CSV/JSON), predictions and raw responses.
R = Path(REPORT_DIR)


def md_table(df, cols=None, digits=3):
    df = df[cols] if cols else df
    head = "| " + " | ".join(map(str, df.columns)) + " |\n|" + "---|" * len(df.columns) + "\n"
    body = "".join("| " + " | ".join(f"{v:.{digits}f}" if isinstance(v, float) else str(v) for v in row) + " |\n"
                   for row in df.itertuples(index=False))
    return head + body


acc_cols = ["dataset", "n", "accuracy", "acc_ci_lo", "acc_ci_hi", "majority_baseline", "balanced_accuracy",
            "nll", "brier", "ece", "overconfidence", "auroc_conf_correct", "tau_coverage", "tau_acc_auto"]
o = overall.iloc[0]
title = f"DeciderBench × {MODEL} — evaluation report ({RUN_ID})"
md = [f"# {title}\n",
      ("> **MOCK RUN — synthetic responses, not real model results.**\n" if MOCK else ""),
      f"Eval rows: **{len(rows)}** · answered: **{ops['rows_answered']}** · requests: {len(done)}/{len(REQUESTS)} · "
      f"model(s): {dict(models)} · input tokens: {in_tok:,} · cost: **${ops['cost_usd']:.4f}** "
      f"(${PRICE_PER_M_INPUT}/1M input) · latency p50/p95: {ops['latency_p50_s']:.3f}s / {ops['latency_p95_s']:.3f}s\n",
      "## Headline\n",
      f"* Accuracy **{o.accuracy:.3f}** (95% CI {o.acc_ci_lo:.3f}–{o.acc_ci_hi:.3f})\n"
      f"* Log loss (NLL) **{o.nll:.3f}** vs uniform {o.nll_uniform:.3f} · Brier **{o.brier:.3f}** "
      f"(article 1/K form {o.brier_article_1_over_K:.3f})\n"
      f"* ECE **{o.ece:.3f}** (equal-mass {o.ece_equal_mass:.3f}, MCE {o.mce:.3f}) · mean confidence "
      f"{o.mean_conf:.3f} → overconfidence gap {o.overconfidence:+.3f}\n"
      f"* Confidence AUROC for correctness **{o.auroc_conf_correct:.3f}** · AURC {o.aurc:.3f}\n"
      f"* Article routing τ = {TAU:.3f}: auto-routes **{o.tau_coverage:.1%}** of decisions at "
      f"{o.tau_acc_auto:.3f} accuracy; expected cost/decision ${o.tau_cost_per_decision:.3f} vs "
      f"${o.tau_cost_all_auto:.3f} all-auto and ${C_HUMAN:.2f} all-human\n",
      "## Accuracy and calibration per dataset\n", md_table(by_dataset, acc_cols),
      "\n## Per primitive\n", md_table(by_primitive, ["primitive"] + KEY_COLS[:-3]),
      "\n## BBH per task\n", md_table(by_bbh_task, ["subset"] + KEY_COLS[:-3]),
      "\n## Score primitive\n", md_table(score_metrics) if len(score_metrics) else "n/a\n",
      "\n## Noul primitive\n", md_table(noul_metrics) if len(noul_metrics) else "n/a\n",
      "\n## Cost-based routing per dataset (τ = %.3f)\n" % TAU, md_table(routing_by_dataset),
      "\n## Threshold sweep (all rows)\n", md_table(sweep),
      "\n## Figures\n![](figures/reliability_overall.png)\n![](figures/reliability_by_dataset.png)\n"
      "![](figures/accuracy_coverage.png)\n",
      "\n## Operations\n```json\n" + json.dumps(ops, indent=1) + "\n```\n",
      "\n## Notes\n* Accuracy for Score uses the most probable level; Noul uses p ≥ 0.5.\n"
      "* ECE/AUROC use top-label confidence p_max from the returned distribution; the API `confidence` field "
      "(Choice/Score only) is reported separately as `auroc_api_confidence` in metrics.json.\n"
      "* Rows sharing a state were sent in one request (questions are evaluated independently per the API docs).\n"
      "* Known source-label caveats (RAGTruth class imbalance and under-annotation, noisy ContractNLI neutral) "
      "are documented in DeciderBench_stats.json.\n"]
(R / "report.md").write_text("".join(md), encoding="utf-8")

imgs = "".join(f'<h3>{p.stem}</h3><img style="max-width:100%" src="data:image/png;base64,'
               f'{base64.b64encode(p.read_bytes()).decode()}"/>' for p in sorted(FIG.glob("*.png")))
html_tables = "".join(f"<h2>{name}</h2>" + df.to_html(index=False, float_format=lambda x: f"{x:.3f}", border=0)
                      for name, df in [("Per dataset", by_dataset[acc_cols]), ("Per primitive", by_primitive),
                                       ("BBH per task", by_bbh_task), ("Score primitive", score_metrics),
                                       ("Noul primitive", noul_metrics), ("Routing per dataset", routing_by_dataset),
                                       ("Threshold sweep", sweep)])
headline = md[4].replace("* ", "<li>").replace("\n", "</li>").replace("**", "")
(R / "report.html").write_text(
    f"<!doctype html><meta charset='utf-8'><title>{title}</title><style>body{{font-family:system-ui,sans-serif;"
    f"margin:2rem;max-width:1400px}}table{{border-collapse:collapse;font-size:12px}}td,th{{padding:3px 8px;"
    f"border-bottom:1px solid #ddd;text-align:right}}th{{background:#f4f4f4}}</style><h1>{title}</h1>"
    f"{'<p><b>MOCK RUN — synthetic responses, not real model results.</b></p>' if MOCK else ''}"
    f"<p>{md[2]}</p><ul>{headline}</ul>{html_tables}<h2>Figures</h2>{imgs}"
    f"<h2>Operations</h2><pre>{json.dumps(ops, indent=1)}</pre>", encoding="utf-8")

metrics_json = {"run_id": RUN_ID, "mock": MOCK, "model": MODEL, "tau": TAU,
                "costs": {"auto": C_AUTO, "human": C_HUMAN, "error": C_ERROR}, "ops": ops,
                "overall": overall.to_dict("records")[0], "by_primitive": by_primitive.to_dict("records"),
                "by_dataset": by_dataset.to_dict("records"), "by_dataset_primitive": by_dataset_primitive.to_dict("records"),
                "by_bbh_task": by_bbh_task.to_dict("records"), "by_subset": by_subset.to_dict("records"),
                "score": score_metrics.to_dict("records"), "noul": noul_metrics.to_dict("records"),
                "threshold_sweep": sweep.to_dict("records")}
(R / "metrics.json").write_text(json.dumps(metrics_json, indent=1, default=lambda x: None if pd.isna(x) else x),
                                encoding="utf-8")
for name, df in [("per_dataset", by_dataset), ("per_primitive", by_primitive), ("per_dataset_primitive",
                 by_dataset_primitive), ("per_bbh_task", by_bbh_task), ("score_metrics", score_metrics),
                 ("noul_metrics", noul_metrics), ("routing_per_dataset", routing_by_dataset), ("threshold_sweep", sweep)]:
    df.to_csv(R / f"{name}.csv", index=False)
pred.to_parquet(R / "predictions.parquet", index=False)
shutil.copy(RESPONSES_PATH, R / RESPONSES_PATH.name)
print("report written to", R.resolve())
for p in sorted(R.rglob("*")):
    if p.is_file():
        print(f"  {p.relative_to(R)}  {p.stat().st_size:,} B")

report written to /Users/dhruvilpatel/Downloads/D/decision_model_bench/jev_eval_runs/jev-latest/report_20261002T053634Z
  figures/accuracy_coverage.png  119,152 B
  figures/reliability_by_dataset.png  244,137 B
  figures/reliability_overall.png  98,691 B
  metrics.json  121,574 B
  noul_metrics.csv  4,514 B
  per_bbh_task.csv  6,020 B
  per_dataset.csv  6,495 B
  per_dataset_primitive.csv  7,963 B
  per_primitive.csv  2,328 B
  predictions.parquet  235,326 B
  report.html  655,353 B
  report.md  13,259 B
  responses.jsonl  3,310,394 B
  routing_per_dataset.csv  1,136 B
  score_metrics.csv  220 B
  threshold_sweep.csv  1,237 B


In [17]:
# Summary
display(fmt(by_dataset[["dataset", "n", "accuracy", "acc_ci_lo", "acc_ci_hi", "nll", "brier", "ece", "auroc_conf_correct"]]))
print(f"OVERALL accuracy={o.accuracy:.4f}  NLL={o.nll:.4f}  Brier={o.brier:.4f}  ECE={o.ece:.4f}  "
      f"cost=${ops['cost_usd']:.4f}{'  [MOCK]' if MOCK else ''}")

,dataset,n,accuracy,acc_ci_lo,acc_ci_hi,nll,brier,ece,auroc_conf_correct
0,BBH,700,0.900,0.876,0.920,0.267,0.132,0.034,0.909
1,Belebele,500,0.978,0.961,0.988,0.257,0.034,0.010,0.934
2,CIRCA,500,0.902,0.873,0.925,0.295,0.150,0.032,0.877
3,ContractNLI,500,0.838,0.803,0.868,1.038,0.256,0.067,0.769
4,FinancialPhraseBank,1000,0.921,0.903,0.936,0.194,0.118,0.017,0.896
5,JevBench-Hard,293,0.672,0.617,0.724,0.755,0.426,0.096,0.729
6,JudgeBench,300,0.790,0.740,0.832,0.450,0.288,0.034,0.752
7,RAGTruth,1494,0.777,0.755,0.797,0.504,0.322,0.043,0.690
8,TabFact,500,0.932,0.906,0.951,0.180,0.102,0.015,0.880
9,TruthfulQA-Binary,500,0.964,0.944,0.977,0.197,0.056,0.018,0.866


OVERALL accuracy=0.8735  NLL=0.3915  Brier=0.1830  ECE=0.0200  cost=$0.1754
